In [21]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster, TimestampedGeoJson

database_path = Path("../data/db/edition.sqlite")

def read_df(sql: str, params=()):
    with sqlite3.connect(database_path) as conn:
        return pd.read_sql_query(sql, conn, params=params)

In [22]:
sql_site_period = """
SELECT
  s.id AS site_id,
  s.code AS site_code,
  s.name AS site_name,
  v.name AS village_name,
  c.latitude,
  c.longitude,
  pc.period AS period,
  SUM(COALESCE(pcs.sherd_count, 0)) AS sherds
FROM pottery_class_site pcs
JOIN pottery_class pc ON pc.id = pcs.pottery_class_id
JOIN site s ON s.id = pcs.site_id
LEFT JOIN village v ON v.id = s.village_id
JOIN coords c ON c.site_id = s.id
WHERE c.latitude IS NOT NULL
  AND c.longitude IS NOT NULL
  AND c.latitude BETWEEN -90 AND 90
  AND c.longitude BETWEEN -180 AND 180
GROUP BY
  s.id, s.code, s.name, v.name, c.latitude, c.longitude, pc.period
ORDER BY s.code, pc.period;
"""

df_sp = read_df(sql_site_period)
df_sp.head(), df_sp.shape


(   site_id site_code         site_name       village_name   latitude  \
 0        1    CS.1.1  AI-Fulayj CS.1.1  AL-FULAYJ VILLAGE  22.888825   
 1        1    CS.1.1  AI-Fulayj CS.1.1  AL-FULAYJ VILLAGE  22.888825   
 2        4    CS.1.4  AI-Fulayj CS.1.4  AL-FULAYJ VILLAGE  22.866359   
 3        4    CS.1.4  AI-Fulayj CS.1.4  AL-FULAYJ VILLAGE  22.866359   
 4        4    CS.1.4  AI-Fulayj CS.1.4  AL-FULAYJ VILLAGE  22.866359   
 
    longitude                period  sherds  
 0  58.061360             L.Islamic       3  
 1  58.061360                 U.Nar       2  
 2  58.050853              Islamic?       1  
 3  58.050853             L.Islamic       3  
 4  58.050853  L.Islamic to Recent?       4  ,
 (152, 8))

In [23]:
df_sp["period"].value_counts().head(20)


period
L.Islamic                 31
LIA                       24
L.Islamic to Recent?      22
Recent                    15
M.Islamic                 14
EIA                       13
U.Nar                     11
Islamic?                   8
W.Suq                      6
M.Islamic to L.Islamic     3
IA                         2
E.Islamic                  2
L.B.A                      1
Name: count, dtype: int64

In [24]:
def map_center(df):
    return [df["latitude"].mean(), df["longitude"].mean()]

def period_palette(periods):
    # einfache, stabile Farbliste (ohne externe libs)
    base = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173"
    ]
    periods = list(periods)
    return {p: base[i % len(base)] for i, p in enumerate(periods)}

def radius_scale(values, r_min=3, r_max=12):
    # robust: log-Skalierung, verhindert extreme Ausreißer
    v = np.array(values, dtype=float)
    v = np.where(v < 0, 0, v)
    v = np.log1p(v)
    if v.max() == v.min():
        return lambda x: (r_min + r_max) / 2
    return lambda x: r_min + (np.log1p(max(x,0)) - v.min()) * (r_max - r_min) / (v.max() - v.min())


In [25]:
periods = sorted(df_sp["period"].dropna().unique())
colors = period_palette(periods)

center = map_center(df_sp)
m_layers = folium.Map(location=center, zoom_start=8, tiles="OpenStreetMap")

# Optional: Cluster pro Layer (hilft bei vielen Punkten)
# Wir legen je Periode einen FeatureGroup an, und darin einen MarkerCluster.
for p in periods:
    fg = folium.FeatureGroup(name=f"Period: {p}", show=(p == periods[0]))
    cluster = MarkerCluster().add_to(fg)

    d = df_sp[df_sp["period"] == p]
    for _, row in d.iterrows():
        popup_html = f"""
        <b>{row['site_code']}</b><br>
        {row['site_name'] or ''}<br>
        Village: {row['village_name'] or ''}<br>
        Period: {row['period']}<br>
        Sherds: {int(row['sherds'])}
        """
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5,
            color=colors[p],
            fill=True,
            fill_color=colors[p],
            fill_opacity=0.7,
            tooltip=f"{row['site_code']} ({p})",
            popup=folium.Popup(popup_html, max_width=350),
        ).add_to(cluster)

    fg.add_to(m_layers)

folium.LayerControl(collapsed=False).add_to(m_layers)
m_layers


In [26]:
period_order = periods  # TODO: hier ggf. manuell fachlich korrekt sortieren
start = pd.Timestamp("2000-01-01")
period_to_time = {p: (start + pd.DateOffset(months=i)).strftime("%Y-%m-%d") for i, p in enumerate(period_order)}
period_to_time


{'E.Islamic': '2000-01-01',
 'EIA': '2000-02-01',
 'IA': '2000-03-01',
 'Islamic?': '2000-04-01',
 'L.B.A': '2000-05-01',
 'L.Islamic': '2000-06-01',
 'L.Islamic to Recent?': '2000-07-01',
 'LIA': '2000-08-01',
 'M.Islamic': '2000-09-01',
 'M.Islamic to L.Islamic': '2000-10-01',
 'Recent': '2000-11-01',
 'U.Nar': '2000-12-01',
 'W.Suq': '2001-01-01'}

In [27]:
features = []
for _, row in df_sp.iterrows():
    t = period_to_time.get(row["period"])
    if not t:
        continue

    # Feature pro Site×Periode (mit Sherd-Menge)
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [row["longitude"], row["latitude"]]},
        "properties": {
            "time": t,
            "popup": f"{row['site_code']} | {row['period']} | Sherds: {int(row['sherds'])}",
            "style": {"color": colors.get(row["period"], "#000000")},
            "icon": "circle",
            "iconstyle": {
                "fillColor": colors.get(row["period"], "#000000"),
                "fillOpacity": 0.7,
                "stroke": "true",
                "radius": 5
            }
        }
    })

geojson = {"type": "FeatureCollection", "features": features}

m_anim = folium.Map(location=center, zoom_start=8, tiles="OpenStreetMap")

TimestampedGeoJson(
    geojson,
    period="P1M",          # 1 Monat pro Schritt (synthetisch)
    add_last_point=True,
    auto_play=False,
    loop=False,
    max_speed=2,
    loop_button=True,
    date_options="YYYY-MM",  # zeigt Jahr-Monat (passt zu synthetischen Perioden)
    time_slider_drag_update=True,
).add_to(m_anim)

m_anim
